
# Loan Prediction - Exploratory Data Analysis (EDA)

## Objectives
- Understand data quality and missing values
- Analyze target variable distribution
- Study categorical and numerical features
- Detect outliers and skewness
- Understand relationships with Loan_Status
- Prepare dataset for feature engineering and modeling

### Observation Section
**Write your observations below each section after reviewing the outputs.**


In [ ]:

# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10,6)


In [ ]:

# ============================================
# LOAD DATA
# ============================================

df_loan = pd.read_csv('./datasets/train_ctrUa4K.csv')

print(f'Shape: {df_loan.shape}')
display(df_loan.head())


## 1. Dataset Overview

### Observation:
- 

In [ ]:

display(df_loan.info())
display(df_loan.describe(include='all').T)


## 2. Missing Value Analysis

### Observation:
- 

In [ ]:

missing = pd.DataFrame({
    'Missing Count': df_loan.isnull().sum(),
    'Missing %': round(df_loan.isnull().mean()*100,2)
})

missing = missing[missing['Missing Count']>0].sort_values('Missing Count',ascending=False)

display(
    missing.style.background_gradient(cmap='Reds')
)

plt.figure(figsize=(10,4))
sns.heatmap(df_loan.isnull(), cbar=False)
plt.title('Missing Value Heatmap')
plt.show()


## 3. Target Variable Analysis

### Observation:
- 

In [ ]:

target_dist = df_loan['Loan_Status'].value_counts()

display(target_dist)

plt.figure(figsize=(6,4))
sns.countplot(data=df_loan,x='Loan_Status')
plt.title('Loan Status Distribution')
plt.show()

print(round(df_loan['Loan_Status'].value_counts(normalize=True)*100,2))


## 4. Numerical Feature Analysis

### Observation:
- 

In [ ]:

num_cols = df_loan.select_dtypes(include=np.number).columns

display(df_loan[num_cols].describe().T)

df_loan[num_cols].hist(
    figsize=(15,10),
    bins=30
)
plt.tight_layout()
plt.show()


## 5. Outlier Detection

### Observation:
- 

In [ ]:

for col in num_cols:
    plt.figure(figsize=(8,3))
    sns.boxplot(x=df_loan[col])
    plt.title(f'Boxplot - {col}')
    plt.show()


## 6. Categorical Feature Analysis

### Observation:
- 

In [ ]:

cat_cols = df_loan.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Loan_ID')

for col in cat_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df_loan,x=col,order=df_loan[col].value_counts(dropna=False).index)
    plt.title(col)
    plt.xticks(rotation=45)
    plt.show()

    display(
        df_loan[col].value_counts(dropna=False)
        .reset_index()
    )


## 7. Feature vs Target Analysis

### Observation:
- 

In [ ]:

for col in cat_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df_loan,x=col,hue='Loan_Status')
    plt.xticks(rotation=45)
    plt.title(f'{col} vs Loan Status')
    plt.show()


## 8. Correlation & Relationships

### Observation:
- 

In [ ]:

temp = df_loan.copy()

for col in temp.select_dtypes(include='object').columns:
    if col!='Loan_ID':
        temp[col] = LabelEncoder().fit_transform(temp[col].astype(str))

corr = temp.corr(numeric_only=True)

plt.figure(figsize=(12,8))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)
plt.title('Correlation Matrix')
plt.show()


## 9. Data Cleaning & Feature Engineering

### Observation:
- 

In [ ]:

# Target Encoding
df_loan['Loan_Status'] = df_loan['Loan_Status'].map({'N':0,'Y':1})

# Missing Value Treatment

df_loan['Gender'] = df_loan['Gender'].fillna(df_loan['Gender'].mode()[0])
df_loan['Married'] = df_loan['Married'].fillna(df_loan['Married'].mode()[0])
df_loan['Self_Employed'] = df_loan['Self_Employed'].fillna(df_loan['Self_Employed'].mode()[0])

df_loan['LoanAmount'] = df_loan['LoanAmount'].fillna(df_loan['LoanAmount'].median())
df_loan['Loan_Amount_Term'] = df_loan['Loan_Amount_Term'].fillna(df_loan['Loan_Amount_Term'].median())
df_loan['Credit_History'] = df_loan['Credit_History'].fillna(df_loan['Credit_History'].mode()[0])

# Education Encoding
df_loan['Education'] = df_loan['Education'].map({
    'Not Graduate':0,
    'Graduate':1
})

binary_cols = ['Gender','Married','Self_Employed']

for col in binary_cols:
    le = LabelEncoder()
    df_loan[col] = le.fit_transform(df_loan[col])

# Feature Engineering
df_loan['TotalIncome'] = (
    df_loan['ApplicantIncome'] +
    df_loan['CoapplicantIncome']
)

df_loan['Income_per_Loan'] = (
    df_loan['TotalIncome'] /
    df_loan['LoanAmount']
)

# One Hot Encoding
area_dummies = pd.get_dummies(
    df_loan['Property_Area'],
    prefix='Area',
    drop_first=True,
    dtype=int
)

df_loan = pd.concat([df_loan,area_dummies],axis=1)

display(df_loan.head())


## 10. Final Dataset Summary

### Observation:
- 

In [ ]:

print(df_loan.shape)

display(df_loan.head())

display(
    df_loan.describe().T
    .style.background_gradient(cmap='Blues')
)
